LoRA from Scratch

In [3]:
import math
import torch
import torch.nn as nn

Hyper Parameters

In [4]:
rank = 8
alpha = 2 * rank
dropout = 0.05

LoRA Linear Class

In [6]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: int, dropout: float = dropout):
        super().__init__()
        self.base = base
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        self.dropout = nn.Dropout(dropout)

        for p in self.base.parameters():
            p.requires_grad = False
        
        self.A = nn.Linear(base.in_features, rank, bias=False)
        self.B = nn.Linear(rank, base.out_features, bias=False)

        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)
    
    def forward(self, x):
        return self.base(x) + self.B(self.A(self.dropout(x))) * self.scaling

Replace LoRA Modules

In [7]:
def replace_lora_modules(model: nn.Module, target_names: tuple, rank: int, alpha: int):
    for name, module in list(model.named_modules()):
        if any(name.endswith(t) for t in target_names) and isinstance(module, nn.Linear):
            parent_name, child_name = name.rsplit(".", 1)
            parent = model.get_submodule(parent_name)       
            setattr(parent, child_name, LoRALinear(module, rank, alpha))